<a href="https://www.kaggle.com/code/viktorkondrashov123/practical-nn-les4?scriptVersionId=294425152" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import torch  
from torchvision import datasets, transforms 

In [ ]:
data_dir = "/kaggle/input/leukemia-classification/C-NMC_Leukemia/training_data/fold_0"

In [ ]:
dataset0 = datasets.ImageFolder(data_dir)

In [ ]:
len(dataset0)

In [ ]:
img, label = dataset0[3526]

In [ ]:
img

In [ ]:
dataset0.classes

In [ ]:
data_dir1 = "/kaggle/input/leukemia-classification/C-NMC_Leukemia/training_data/fold_1"
dataset1 = datasets.ImageFolder(data_dir1)

data_dir2 = "/kaggle/input/leukemia-classification/C-NMC_Leukemia/training_data/fold_2"
dataset2 = datasets.ImageFolder(data_dir2)

In [ ]:
dataset0.classes
dataset1.classes
dataset2.classes

In [ ]:
dataset_all = torch.utils.data.ConcatDataset([dataset0, dataset1, dataset2])

In [ ]:
len(dataset_all)

In [ ]:
data_train, data_test = torch.utils.data.random_split(dataset_all,[0.8, 0.2])

In [ ]:
len(data_train)


In [ ]:
len(data_test)


In [ ]:
img, label = data_train[5000]

In [ ]:
img

In [ ]:
train_transform = transforms.Compose(
    [transforms.Resize([200, 200]),
    transforms.CenterCrop(180),
    transforms.RandomRotation(180),
    transforms.ToTensor()
    ]
)

In [ ]:
train_transform(img).shape

In [ ]:
test_transform = transforms.Compose(
    [transforms.Resize([200, 200]),
    transforms.CenterCrop(180),
    transforms.ToTensor()
    ]
)

In [ ]:
test_transform(img).shape

In [ ]:
from torch.utils.data import Dataset,DataLoader

class TransformDataset(Dataset):
    def __init__(self, dataset, transformer):
        super().__init__()
        self.dataset = dataset
        self.transformer = transformer

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        img, y = self.dataset[idx]

        new_img = self.transformer(img)
        
        return new_img, y

In [ ]:
train_data = TransformDataset(data_train,train_transform)

In [ ]:
test_data = TransformDataset(data_test,test_transform)

In [ ]:
len(test_data)

In [ ]:
test_data[2100]

In [ ]:
train_loader = DataLoader(train_data, batch_size=256)

In [ ]:
test_loader = DataLoader(test_data, batch_size=256)


In [ ]:
for img,idx in test_loader:
     print(img.shape)
     print(idx.shape) 


In [ ]:
import matplotlib.pyplot as plt

for i in range(3):  # Show 3 images

    # Get the image data (tensor) and convert it back to a NumPy array for manipulation
    img, y = train_data[i]
    img = img.numpy()
    
    # Convert the color channels from (channels, height, width) to (height, width, channels) for pyplot
    img = img.transpose((1, 2, 0))
    print(img.shape)
    
    # Get the label name from the dataset class labels
    label = dataset0.classes[y]

    # Plot the image with a title (including label name)
    plt.imshow(img)
    plt.title(f"Label {label}")
    plt.show()

In [ ]:
from torchvision.utils import make_grid

loader = torch.utils.data.DataLoader(train_data, shuffle=True, batch_size=32)

  
batch, labels = next(iter(loader))

grid = make_grid(batch).permute(1, 2, 0) # результатом є тензор

plt.imshow(grid)

In [ ]:
3*60*60

In [ ]:
# from torch import nn

# model = nn.Sequential(
#     nn.Flatten(),
#     nn.Linear(10800, 64),
#     nn.ReLU(),
#     nn.Linear(64, 32),
#     nn.ReLU(),
#     nn.Linear(32, 16),
#     nn.ReLU(),
#     nn.Linear(16, 2)
# )

# device = "cuda"
# model = model.to(device)

from torch import nn

model = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding='same'),# розмір буде 8,180,180
    nn.ReLU(),
    nn.MaxPool2d(2,2), #8, 90 , 90
    nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding='same'),#16, 90, 90
    nn.ReLU(), # 16, 90, 90
    nn.MaxPool2d(2, 2), # 16, 45, 45
    nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding='same'), #32, 45, 45
    nn.ReLU(), #32, 45, 45
    nn.MaxPool2d(2, 2), # 32, 22, 22

    nn.Flatten(), # 32*22*22 - 15488
    nn.Linear(15488, 16),
    nn.ReLU(),
    nn.Linear(16, 2)
)

device = 'cuda'
model = model.to(device)

In [ ]:
32*22*22

In [ ]:
for imgs, labels in train_loader:
    break

In [ ]:
imgs.shape

In [ ]:
res = model(imgs)
res.shape

In [ ]:
# функция ошибки для задачи многоклассовой классификации
loss_fn = nn.CrossEntropyLoss()

# оптимизатор Adam для обучения нейронной сети
optimizer = torch.optim.Adam(
    model.parameters(),   # параметры модели
    lr=0.001
)

# список для сохранения значений loss во время обучения
loss_list = []
loss_test_list = []

# цикл обучения по эпохам
for i in range(3):

    # перебор батчей из обучающего даталоадера
    for imgs, labels in train_loader:
        imgs = imgs.to(device)        # перенос изображений на GPU / CPU
        labels = labels.to(device)    # перенос меток на GPU / CPU

        result = model(imgs)          # прямой проход через модель
        loss = loss_fn(result, labels)  # вычисление функции потерь
        print(f"loss: {loss}")

        loss.backward()               # вычисление градиентов
        optimizer.step()              # обновление весов модели
        optimizer.zero_grad()         # очистка градиентов

        loss_list.append(loss.cpu().item())  # сохранение значения loss

    #test data
    for imgs, labels in train_loader:
        imgs = imgs.to(device)        # перенос изображений на GPU / CPU
        labels = labels.to(device)    # перенос меток на GPU / CPU

        result = model(imgs)          # прямой проход через модель
        loss = loss_fn(result, labels)  # вычисление функции потерь
        print(f"loss: {loss}")

        

        loss_test_list.append(loss.cpu().item())  # сохранение значения loss


In [ ]:
import matplotlib.pyplot as plt

new_list = loss_list[20 :]

plt.plot(new_list)



In [ ]:
plt.plot(loss_test_list)

In [ ]:
imgs.shape

In [ ]:
img, label = train_data[100]
img = img.unsqueeze(0)
img.shape

In [ ]:
img = img.to(device)
prediction = model(img)
print(label)
print(nn.Softmax()(prediction))

In [ ]:
result = model(imgs)

In [ ]:
res1 = result.argmax(dim=1)

In [ ]:
labels

In [ ]:
res2 = res1 == labels

In [ ]:
res2.sum()

In [ ]:
67/81

In [ ]:
count = 0



for imgs, labels in train_loader:
    imgs = imgs.to(device)
    labels = labels.to(device)
    res = model(imgs)
    res = res.argmax(dim=1)
    res2 = res == labels
    summa = res2.sum()
    count += summa.cpu().item()

print(count) 

In [ ]:
len(train_data) 

In [ ]:
6829/8529